# Simulacao e Modelagem Matematica de Problemas

## Live 4 - Applied Math for Data Science

Nesta aula, vamos transformar um problema real em um modelo matematico simples e depois testar esse modelo com simulacao.

O caso escolhido sera uma **fila de inferencia** de uma plataforma de IA, em que pedidos chegam ao longo do tempo e o sistema tem capacidade limitada de atendimento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

print('Ambiente pronto para simulacao.')

---

## 1. Qual e o problema?

Imagine uma API de IA que recebe requisicoes por minuto. Cada requisicao precisa ser processada por uma GPU. Se chegam pedidos demais, forma-se uma fila.

Queremos responder perguntas como:
- a fila cresce indefinidamente?
- qual e o tempo medio de espera?
- em quais cenarios a operacao satura?

### Variaveis do modelo

- `chegadas_t`: numero de requisicoes que chegam no minuto `t`
- `capacidade_t`: quantas requisicoes conseguimos processar no minuto `t`
- `fila_t`: requisicoes pendentes ao final do minuto `t`

A equacao de atualizacao mais importante e:

$$fila_t = \max(0, fila_{t-1} + chegadas_t - capacidade_t)$$

In [ ]:
def simular_fila(minutos, taxa_chegada, capacidade, seed=None):
    rng = np.random.default_rng(seed)
    chegadas = rng.poisson(taxa_chegada, size=minutos)
    fila = np.zeros(minutos)
    atendidas = np.zeros(minutos)

    fila_atual = 0
    for t in range(minutos):
        total_disponivel = fila_atual + chegadas[t]
        atendidas[t] = min(total_disponivel, capacidade)
        fila_atual = max(0, total_disponivel - capacidade)
        fila[t] = fila_atual

    return chegadas, atendidas, fila

---

## 2. Simulando um cenario base

Vamos rodar um cenario em que a taxa media de chegadas e proxima da capacidade do sistema.

In [ ]:
minutos = 180
taxa_chegada = 11
capacidade = 10

chegadas, atendidas, fila = simular_fila(minutos, taxa_chegada, capacidade, seed=42)

print(f'Chegadas totais: {chegadas.sum()}')
print(f'Atendidas totais: {atendidas.sum():.0f}')
print(f'Fila final: {fila[-1]:.0f}')
print(f'Fila maxima: {fila.max():.0f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(chegadas, color='#7bdff2', linewidth=1.8, label='Chegadas por minuto')
axes[0].axhline(capacidade, color='#ffd166', linestyle='--', linewidth=2, label='Capacidade')
axes[0].set_title('Fluxo de entrada vs capacidade')
axes[0].legend()

axes[1].plot(fila, color='#ff70a6', linewidth=2, label='Tamanho da fila')
axes[1].set_title('Evolucao da fila ao longo do tempo')
axes[1].set_xlabel('Minuto')
axes[1].legend()

plt.tight_layout()
plt.show()

### Leitura do grafico

Quando a taxa de chegada fica acima da capacidade, a fila cresce. Quando o sistema consegue processar mais do que chega, a fila reduz.

---

## 3. Comparando cenarios

Agora vamos comparar tres regimes operacionais:
- carga baixa
- carga media
- carga alta

In [ ]:
cenarios = {
    'Baixa carga': 8,
    'Carga media': 10,
    'Alta carga': 13,
}

resultados = {}
for i, (nome, taxa) in enumerate(cenarios.items()):
    resultados[nome] = simular_fila(minutos, taxa, capacidade, seed=100 + i)

plt.figure(figsize=(12, 5))
cores = ['#90f1b8', '#ffd166', '#ff70a6']
for (nome, (_, _, fila_cenario)), cor in zip(resultados.items(), cores):
    plt.plot(fila_cenario, label=nome, linewidth=2.2, color=cor)

plt.title('Comparando a fila em tres cenarios')
plt.xlabel('Minuto')
plt.ylabel('Tamanho da fila')
plt.legend()
plt.show()

In [ ]:
for nome, (_, _, fila_cenario) in resultados.items():
    print(f'{nome}: fila maxima = {fila_cenario.max():.0f}, fila final = {fila_cenario[-1]:.0f}')

Esse tipo de simulacao ajuda times de engenharia a decidir quando aumentar capacidade, balancear carga ou mudar filas de prioridade.

---

## 4. Rodando varias simulacoes

Uma unica trajetoria nao basta. Em sistemas com aleatoriedade, precisamos repetir a simulacao varias vezes para estimar media e risco.

In [ ]:
def repetir_simulacao(n_execucoes, minutos, taxa_chegada, capacidade):
    filas_finais = []
    filas_maximas = []
    for s in range(n_execucoes):
        _, _, fila = simular_fila(minutos, taxa_chegada, capacidade, seed=s)
        filas_finais.append(fila[-1])
        filas_maximas.append(fila.max())
    return np.array(filas_finais), np.array(filas_maximas)

filas_finais, filas_maximas = repetir_simulacao(300, minutos=180, taxa_chegada=11, capacidade=10)

print(f'Fila final media: {filas_finais.mean():.2f}')
print(f'Fila maxima media: {filas_maximas.mean():.2f}')
print(f'Percentil 90 da fila maxima: {np.percentile(filas_maximas, 90):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].hist(filas_finais, bins=25, color='#7bdff2', alpha=0.85)
axes[0].set_title('Distribuicao da fila final')
axes[0].set_xlabel('Fila final')

axes[1].hist(filas_maximas, bins=25, color='#ff70a6', alpha=0.85)
axes[1].set_title('Distribuicao da fila maxima')
axes[1].set_xlabel('Fila maxima')

plt.tight_layout()
plt.show()

### Interpretacao

Agora nao temos so uma resposta, mas uma distribuicao de respostas possiveis. Isso e essencial para analise de risco.

---

## 5. Exemplo de uso real

Esse tipo de modelagem aparece em varios contextos:

- **APIs de IA**: prever saturacao de GPU e latencia
- **Hospitais**: estimar fila de atendimento
- **Logistica**: simular centros de distribuicao
- **Telecom**: testar trafego em rede antes de um pico
- **Apps de mobilidade**: simular demanda por corridas

---

## 6. Conclusoes

O que fizemos nesta aula:

1. Definimos um problema real
2. Escolhemos variaveis e regras de evolucao
3. Simulamos a dinamica do sistema
4. Comparamos cenarios e medimos risco

Em resumo:
- **Modelagem matematica** organiza o problema
- **Simulacao** testa comportamentos possiveis
- **Estatistica** resume os resultados

Esse trio e muito util para tomar decisoes em sistemas reais com incerteza.